# Semaine 2 — Agent IA (Python pur)

**Objectif :** Construire un agent simple utilisant le pattern Function Calling (mock + mode réel).

**Contenu :** introduction, mock tools, boucle d'interaction, FastAPI skeleton, métriques basiques.

---


## Installation
Exécute dans un terminal:
```
pip install openai fastapi uvicorn python-dotenv requests
```  
_Optionnel_: clients vector DB si nécessaire (pinecone-client, weaviate-client).


In [ ]:
from dotenv import load_dotenv
import os
load_dotenv()
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
print('OPENAI_API_KEY set:', bool(OPENAI_API_KEY))

## Section A — Mock mode (exécutable sans clé API)

Les cellules suivantes fonctionnent sans dépendances externes et simulent la logique d'un agent qui appelle des tools.

In [ ]:
# Mock tools and dispatcher
import json

def get_weather_tool(city, country=None):
    return {"city": city, "country": country or "FR", "temp_c": 16, "condition": "Partly cloudy"}

def create_event_tool(title, start_iso, end_iso, participants=None):
    return {"event_id": "evt_mock_001", "title": title, "start": start_iso, "end": end_iso, "participants": participants or []}

def dispatch_function_call(name, arguments):
    if isinstance(arguments, str):
        try:
            arguments = json.loads(arguments)
        except:
            arguments = {}
    if name == 'get_weather':
        return get_weather_tool(**arguments)
    if name == 'create_calendar_event':
        return create_event_tool(**arguments)
    return {"error": "unknown function"}

# Quick mock demonstration
print(dispatch_function_call('get_weather', {'city':'Paris'}))

## Mock interaction loop
Simule un modèle qui demande l'exécution d'une function_call, exécute l'outil localement et renvoie la sortie.

In [ ]:
# Simulated interaction
history = [
    {'role':'system','content':'Tu es un assistant qui peut appeler des fonctions.'},
    {'role':'user','content':'Peux-tu créer un rendez-vous demain à 10h ?'}
]
# Model decides to call function (mock)
model_fc = {'name':'create_calendar_event','arguments':{'title':'Point suivi','start_iso':'2025-10-20T10:00:00','end_iso':'2025-10-20T11:00:00','participants':['bob@example.com']}}
result = dispatch_function_call(model_fc['name'], model_fc['arguments'])
# Add function result to history
history.append({'role':'function','name':model_fc['name'],'content':json.dumps(result)})
print('History after call:', history)


## Section B — Mode réel (OpenAI)

Les cellules ci-dessous montrent comment appeler l'API OpenAI en mode réel. **Ne lance** que si tu as `openai` installé et `OPENAI_API_KEY` configurée.

In [ ]:
try:
    import openai
except Exception:
    openai = None

import json, os

def call_openai_chat(messages, functions=None):
    if openai is None:
        raise RuntimeError('openai package not installed. pip install openai')
    openai.api_key = os.getenv('OPENAI_API_KEY')
    resp = openai.ChatCompletion.create(
        model='gpt-4o-mini',
        messages=messages,
        functions=functions,
        function_call='auto',
        temperature=0.2,
        max_tokens=800
    )
    return resp

# Example wrapper (do not run without key)
print('call_openai_chat wrapper ready')

## FastAPI skeleton (endpoint `/chat`)
Modèle simple d'API exposant l'agent. A utiliser pour wrapper la logique d'appel au modèle + dispatch des functions.

In [ ]:
# FastAPI skeleton - save in a file to run with uvicorn
from fastapi import FastAPI
from pydantic import BaseModel
import json

app = FastAPI()
class ChatRequest(BaseModel):
    user_input: str

@app.post('/chat')
async def chat(req: ChatRequest):
    # Build messages
    messages = [{'role':'system','content':'You are an assistant that can call functions.'}, {'role':'user','content':req.user_input}]
    # In mock mode, simulate
    # In real mode, call_openai_chat(messages, functions)
    return {'mock':'This endpoint is a template - integrate call_openai_chat here.'}

print('FastAPI skeleton cell - save as script to run with uvicorn')

## Exercices proposés
- Étendre les tools (search, docs retrieval)
- Ajouter gestion des sessions et stockage minimal (fichier JSON)
- Écrire des tests unitaires pour dispatcher

Fin du notebook S2 Agent Python (mock + API mode)